# Pointwise dynamics — per-grid-cell analysis (selected windows)

For each ocean pixel **(i, j)** and each selected catalog window, compute:

| Metric | Symbol | Output |
|--------|--------|--------|
| Autocorrelation curve | ρ_ij(τ) | `arrays/{field}_pointwise.npz` → `rho` |
| Integral timescale | T_ij | spatial map + stats |
| First lag ρ < 0.7 / 0.5 / 0.3 | τ_ij | spatial maps |
| Mutual information | I_ij(τ) | Gaussian proxy from ρ (fast) or optional binned |
| Dominant oscillation period | days | from temporal power spectrum |

**Data:** reuses pool12 export — **no re-export**.

| Item | Path |
|------|------|
| Code | [`pointwise_dynamics.py`](pointwise_dynamics.py) |
| Domain-mean sibling | [`Dynamic_data_selected_windows.ipynb`](Dynamic_data_selected_windows.ipynb) |
| Outputs | `Dynamic/pointwise_data/window_{W:02d}_{export_key}/` |

In [ ]:
import json
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
DYNAMIC_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Dynamic" else NOTEBOOK_DIR / "Dynamic"
ROOT = DYNAMIC_DIR.parent
os.chdir(ROOT)
for _p in (ROOT, DYNAMIC_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import pointwise_dynamics as pwd
import window_dynamics as wd
importlib.reload(wd)
importlib.reload(pwd)

# --- user knobs ---
SELECT_WINDOWS = [7, 8]              # 1..12
FIELDS = ("u", "v", "speed")         # speed = √(u²+v²)
MAX_LAG_DAYS = 30
MI_MODE = "gaussian"                 # "gaussian" (fast) or "binned" (slow, exact histogram MI)
MI_BINS = 16
MI_LAG_STRIDE = 3                    # only used when MI_MODE="binned"

DATA_PREFIX = ROOT / "Shalow" / "data" / "select_window_12" / "sample_13_22_uv_dt360"
OUT_ROOT = DYNAMIC_DIR / "pointwise_data"

catalog = wd.load_window_catalog(DATA_PREFIX)
print("Catalog:")
for entry in catalog["windows"]:
    w = int(entry["slot"]) + 1
    mark = " <--" if w in SELECT_WINDOWS else ""
    print(f"  W{w:02d} {entry['export_key']}{mark}")
print(f"\nSELECT_WINDOWS={SELECT_WINDOWS}  FIELDS={FIELDS}  MI_MODE={MI_MODE}")

In [ ]:
summaries = pwd.analyze_selected_windows_pointwise(
    SELECT_WINDOWS,
    data_prefix=DATA_PREFIX,
    out_root=OUT_ROOT,
    fields=FIELDS,
    max_lag_days=MAX_LAG_DAYS,
    mi_mode=MI_MODE,
    mi_bins=MI_BINS,
    mi_lag_stride=MI_LAG_STRIDE,
)

for s in summaries:
    u_stats = s["fields"]["u"]["statistics"]["integral_timescale"]
    print(
        f"W{s['window']:02d} {s['export_key']:28s} | "
        f"u T_int median={u_stats['percentiles']['50']:.1f}d "
        f"(p5={u_stats['percentiles']['5']:.1f}, p95={u_stats['percentiles']['95']:.1f})"
    )

In [ ]:
from IPython.display import Image, display

for s in summaries:
    out_dir = OUT_ROOT / f"window_{s['window']:02d}_{s['export_key']}"
    for field in FIELDS:
        p = out_dir / "plots" / f"01_{field}_spatial_maps.png"
        print(f"\n=== W{s['window']:02d} {field} spatial maps ===")
        display(Image(filename=str(p)))

In [ ]:
# Compare domain-median ρ curves across windows (u field)
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 4))
for s in summaries:
    curve = s["fields"]["u"]["statistics"]["autocorrelation_domain_median_curve"]
    lags = np.arange(1, len(curve) + 1)
    ax.plot(lags, curve, "o-", ms=3, label=f"W{s['window']:02d} {s['export_key']}")
ax.axhline(0.7, ls="--", color="gray", alpha=0.5)
ax.axhline(0.5, ls="--", color="gray", alpha=0.4)
ax.axhline(0.3, ls="--", color="gray", alpha=0.3)
ax.set_xlabel("Lag τ (days)")
ax.set_ylabel("median ρ_ij(τ)")
ax.set_title("Pointwise u: domain-median autocorrelation by window")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_ROOT / "compare_median_rho_u.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved", OUT_ROOT / "compare_median_rho_u.png")